# 🫁 ChestAI: Deep Learning Radiology Intelligence & Explainability
### DenseNet-121 Multi-Label Screening, Temperature Scaling Calibration, and Grad-CAM Attention Maps

**Author:** ChestAI Research Team  
**Dataset Benchmarks:** Stanford CheXpert (`CheXpert-v1.0-small`) & NIH ChestX-ray14 (`Data_Entry_2017_v2020`)  
**Target Pathologies:** Cardiomegaly, Pleural Effusion, Atelectasis, Pulmonary Edema  

---
### Overview
This research notebook demonstrates the end-to-end radiology workflow:
1. **Model Loading & Checkpoint Verification:** DenseNet-121 multi-label classifier
2. **Radiographic Inference & Probability Calibration:** Post-hoc Temperature Scaling ($T=0.05$)
3. **Cardiothoracic Ratio (CTR) Estimation:** Automated anatomical calipers for cardiac silhouette
4. **Clinical Diagnostic Syndrome Synthesis:** Multi-finding pattern synthesis & management
5. **Grad-CAM Explainability:** Class activation mapping showing anatomical focus
6. **Dataset Shift Analysis:** Internal validation vs. external zero-shot generalization

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import yaml
from PIL import Image
from torchvision import transforms

# Ensure project root on path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.train import build_model
from src.gradcam import GradCAM, denormalize, overlay_heatmap
from src.clinical_engine import estimate_ctr, diagnose_syndrome
from src.report_generator import compute_triage_tier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Hardware Device: {device}")
LABEL_NAMES = ["Cardiomegaly", "Pleural Effusion", "Atelectasis", "Edema"]
LABEL_ICONS = ["🫀", "💧", "🫁", "🌊"]

## 1. Load Trained DenseNet-121 Classifier
We load the model weights from `results/checkpoints/best_model.pth` and the post-hoc calibration temperature.

In [ ]:
cfg = {'model': {'backbone': 'densenet121', 'pretrained': False}}
model = build_model(cfg, num_classes=4)

ckpt_path = ROOT / "results" / "checkpoints" / "best_model.pth"
if ckpt_path.exists():
    model.load_state_dict(torch.load(str(ckpt_path), map_location=device))
    print(f"✓ Successfully loaded model checkpoint from {ckpt_path}")
model.eval().to(device)

cal_path = ROOT / "results" / "checkpoints" / "best_model_calibrated.pth"
temp = 0.05
if cal_path.exists():
    cal_data = torch.load(str(cal_path), map_location=device)
    if isinstance(cal_data, dict) and 'temperature' in cal_data:
        temp = float(cal_data['temperature'])
print(f"✓ Optimal Fitted Temperature: T = {temp:.4f}")

## 2. Radiograph Preprocessing & Inference Pipeline

In [ ]:
tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

sample_path = ROOT / "data" / "samples" / "sample_pulmonary_edema.jpg"
img_pil = Image.open(sample_path).convert("RGB")
img_t = tf(img_pil).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(img_t)
    raw_probs = torch.sigmoid(logits).cpu().numpy()[0]
    cal_probs = torch.sigmoid(logits / temp).cpu().numpy()[0]

print("Pathology Probabilities:")
for icon, name, rp, cp in zip(LABEL_ICONS, LABEL_NAMES, raw_probs, cal_probs):
    print(f"  {icon} {name:<18} Raw: {rp:>6.1%}  -->  Calibrated (T={temp}): {cp:>6.1%}")

## 3. Automated Cardiothoracic Ratio (CTR) Calipers
Clinical diagnosis of cardiomegaly relies on quantitative CTR ($C/T$). Normal value is $< 50\%$. Values $> 65\%$ represent severe cardiac enlargement.

In [ ]:
ctr_result = estimate_ctr(img_pil)
print(f"Estimated CTR: {ctr_result['ctr']:.1%}")
print(f"Clinical Grade: {ctr_result['classification']}")
print(f"Note: {ctr_result['clinical_note']}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(ctr_result["annotated_pil"])
ax.set_title(f"Digital Calipers: CTR = {ctr_result['ctr']:.1%} ({ctr_result['severity']})", color="navy", fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Multi-Pathology Diagnostic Syndrome Synthesis
In real clinical settings, diseases present as diagnostic syndromes (e.g. Congestive Heart Failure Triad, Dependent Collapse with Fluid).

In [ ]:
syndrome = diagnose_syndrome(cal_probs.tolist(), LABEL_NAMES, threshold=0.5)
print(f"Syndrome: {syndrome['syndrome']}")
print(f"Acuity: {syndrome['acuity']}")
print(f"ICD-10: {syndrome['icd10']}")
print(f"Summary: {syndrome['summary']}")
print("\nTop Differentials:")
for d in syndrome["differentials"]:
    print(f"  • {d}")
print("\nActionable Recommendations:")
for r in syndrome["recommendations"]:
    print(f"  • {r}")

## 5. Grad-CAM Explainability Heatmaps
Grad-CAM calculates gradients of class scores with respect to the final convolutional feature maps (DenseNet-121 `denseblock4.denselayer16.conv2`), revealing the anatomical attention regions.

In [ ]:
cam = GradCAM(model, backbone="densenet121")
img_np = denormalize(img_t.squeeze(0))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (ax, icon, name) in enumerate(zip(axes, LABEL_ICONS, LABEL_NAMES)):
    hm = cam(img_t, i, device)
    overlay = overlay_heatmap(img_np, hm, alpha=0.45)
    ax.imshow(overlay)
    ax.set_title(f"{icon} {name}\nConf: {cal_probs[i]:.1%}", fontsize=11, fontweight="bold")
    ax.axis("off")

plt.suptitle("Simultaneous 4-Pathology Grad-CAM Attention Matrix", fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

## 6. External Generalization & Dataset Shift Analysis
Comparison of internal held-out validation (CheXpert) vs. zero-shot external test set (NIH ChestX-ray14) with non-parametric bootstrap 95% confidence intervals.

In [ ]:
metrics_file = ROOT / "results" / "metrics" / "external_eval.json"
if metrics_file.exists():
    import json
    with open(metrics_file) as f:
        ext_eval = json.load(f)
    
    nih = ext_eval.get("nih_results", {})
    int_val = ext_eval.get("internal_val_aurocs", {})
    drop = ext_eval.get("drop", {})
    
    print(f"{'Label':<18} | {'Internal Val AUROC':<20} | {'NIH Zero-Shot AUROC (95% CI)':<30} | {'Drop':<10}")
    print("-" * 85)
    for name in LABEL_NAMES:
        iv = int_val.get(name, 1.0)
        nr = nih.get(name, {}).get("auroc", 1.0)
        ci = nih.get(name, {}).get("ci_95", [1.0, 1.0])
        d = drop.get(name, 0.0)
        print(f"{name:<18} | {iv:<20.4f} | {nr:.4f} [{ci[0]:.4f} - {ci[1]:.4f}]           | {d:+.4f}")
else:
    print("Run `python -m src.evaluate_external` to produce benchmark metrics.")